# Saved-session data stream

Consumes the frozen audited corpus. No simulation, model, optimizer or forecast evaluation.
Use the BookSpace `.venv` kernel. Each batch contains unique normalized histories,
raw future paths, pair mappings and scaled future distances.

The small queue below is a visible smoke check. For sustained loading, use the
benchmarked queue size in `docs/training_data.md`. Keep checkpoints between runs;
creating a fresh stream with the same seed intentionally restarts the same sequence.

In [1]:
from mbo_lab.paths import DATA
from mbo_lab.corpus import read_corpus
from mbo_lab.preprocessing import read_preprocessing
from mbo_lab.stream import PairStream

artifact_dir = DATA / "processed/abides/training-v1"
corpus, source_root = read_corpus(artifact_dir / "corpus.json")
preprocessing = read_preprocessing(artifact_dir / "preprocessing.json", corpus)
seed = 123
workers = 4
memory_gib = 6
queue_pairs = 128
batch_pairs = 32
print(f"Frozen sessions: {len(corpus['sessions']):,}")
print(f"Training rows used to fit features: {preprocessing['features']['training_history_rows']:,}")

Frozen sessions: 1,102
Training rows used to fit features: 39,081,051


In [2]:
import time
import numpy as np

settings = dict(seed=seed, workers=workers, memory_bytes=memory_gib * 1024**3,
                queue_pairs=queue_pairs, batch_pairs=batch_pairs)
checkpoint = artifact_dir / "notebook-stream.checkpoint.json"
started = time.perf_counter()
with PairStream(corpus, source_root, preprocessing, **settings) as stream:
    sample = next(stream)
    stream.checkpoint(checkpoint)
    expected_next = next(stream)
    print(f"X {sample.X.shape} {sample.X.dtype}; Y {sample.Y.shape} {sample.Y.dtype}")
    print(f"pairs {sample.pairs.shape}; D {sample.D.shape}")
    print(f"First pair IDs: {sample.pair_ids[:5].tolist()}")
    print(f"First source IDs: {sample.sample_ids[:2]}")
    print(f"Ready in {time.perf_counter() - started:.1f}s")
    report = stream.report()
    print({k: report[k] for k in ('pairs', 'sessions_seen', 'cache_loads', 'cache_hits',
                                  'queue_budget_bytes', 'cache_peak_bytes')})

X (64, 256, 86) float32; Y (64, 128) float64
pairs (32, 2); D (32,)
First pair IDs: [522243285886751, 564354497170068, 623522622681880, 607603279782345, 701348424226984]
First source IDs: (('2021-03-09', 43114), ('2021-04-02', 6990))
Ready in 8.5s
{'pairs': 64, 'sessions_seen': 120, 'cache_loads': 223, 'cache_hits': 0, 'queue_budget_bytes': 22822912, 'cache_peak_bytes': 5030601840}


In [ ]:
with PairStream(corpus, source_root, preprocessing, **settings) as resumed:
    resumed.restore(checkpoint)
    actual_next = next(resumed)
    for name in ("X", "Y", "pairs", "pair_ids", "D", "D_raw"):
        np.testing.assert_array_equal(getattr(actual_next, name), getattr(expected_next, name))
    assert actual_next.sample_ids == expected_next.sample_ids
print("Checkpoint reload reproduces the next batch exactly.")

Checkpoint reload reproduces the next batch exactly.
